# Department Server Metrics Assignment
## AG News Pruning Experiment — DistilBERT & GPT-2

**Server:** IIT Bhubaneswar Department Server (10.10.39.108 — NVIDIA L40S GPU)

**This notebook produces the Department_Server_Metrics_Assignment.xlsx with all required metrics:**

| Category | Metrics |
|---|---|
| Performance | Accuracy, F1 Score, Precision, Recall |
| Computational Efficiency | Inference Latency, Throughput, Training Time, Speedup |
| System Efficiency | GPU Memory Usage, GPU Utilization, Energy Consumption |
| Compression | Compression Ratio, Model Size, Sparsity, Bit-width, Param Reduction |

**Models:** DistilBERT-base-uncased, GPT-2
**Dataset:** AG News (4-class classification)
**Pruning Methods:** Baseline, Magnitude, Random, Movement, Drift-Basic, Drift-Norm, Drift-NoNorm, Drift-Hybrid
**Pruning Ratios:** 10%, 20%, 30%, 40%, 50%, 60%, 70%


In [ ]:
!pip install torch transformers datasets scikit-learn tqdm pynvml pandas openpyxl -q


In [ ]:
import torch, copy, random, time, csv, os
import numpy as np
import torch.nn as nn
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import pynvml

# ── GPU monitoring ────────────────────────────────────────
pynvml.nvmlInit()
gpu_handle = pynvml.nvmlDeviceGetHandleByIndex(0)

def get_gpu_memory_mb():
    return pynvml.nvmlDeviceGetMemoryInfo(gpu_handle).used / 1024**2

def get_gpu_util():
    return pynvml.nvmlDeviceGetUtilizationRates(gpu_handle).gpu

def get_power_watts():
    return pynvml.nvmlDeviceGetPowerUsage(gpu_handle) / 1000.0

# ── Config ────────────────────────────────────────────────
SEED   = 42
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'GPU: {pynvml.nvmlDeviceGetName(gpu_handle)}')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


In [ ]:
# ── METRICS HELPERS ───────────────────────────────────────

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def count_nonzero(model):
    return sum(torch.count_nonzero(p).item() for p in model.parameters())

def model_size_mb(model):
    return count_params(model) * 4 / 1024**2

def effective_size_mb(model):
    return count_nonzero(model) * 4 / 1024**2

def evaluate(model, loader, base_params, base_size_mb, base_latency=None):
    model.eval()
    preds, labels, power = [], [], []
    torch.cuda.synchronize()
    gpu_mem_before = get_gpu_memory_mb()
    start = time.time()
    with torch.no_grad():
        for b in loader:
            out = model(
                input_ids=b['input_ids'].to(device),
                attention_mask=b['attention_mask'].to(device)
            )
            preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
            labels.extend(b['label'].numpy())
            power.append(get_power_watts())
    torch.cuda.synchronize()
    elapsed = time.time() - start
    gpu_mem_after = get_gpu_memory_mb()

    latency    = elapsed / len(loader)
    throughput = len(loader.dataset) / elapsed
    speedup    = (base_latency / latency) if base_latency else 1.0
    nz         = count_nonzero(model)
    total      = count_params(model)
    sparsity   = (total - nz) / total
    eff_size   = effective_size_mb(model)
    comp_ratio = base_size_mb / eff_size if eff_size > 0 else 0
    param_red  = (base_params - nz) / base_params
    avg_power  = np.mean(power)
    energy_j   = avg_power * elapsed

    return {
        'accuracy':           round(accuracy_score(labels, preds), 4),
        'f1':                 round(f1_score(labels, preds, average='macro', zero_division=0), 4),
        'precision':          round(precision_score(labels, preds, average='macro', zero_division=0), 4),
        'recall':             round(recall_score(labels, preds, average='macro', zero_division=0), 4),
        'latency_s':          round(latency, 4),
        'throughput_sps':     round(throughput, 2),
        'speedup':            round(speedup, 4),
        'gpu_memory_mb':      round(gpu_mem_after, 1),
        'gpu_util_pct':       get_gpu_util(),
        'energy_joules':      round(energy_j, 2),
        'model_size_mb':      round(model_size_mb(model), 2),
        'eff_size_mb':        round(eff_size, 2),
        'sparsity_pct':       round(sparsity * 100, 2),
        'compression_ratio':  round(comp_ratio, 4),
        'param_reduction_pct':round(param_red * 100, 2),
        'nonzero_params':     nz,
    }

print('Metrics helpers defined.')


In [ ]:
# ── PRUNING METHODS ───────────────────────────────────────

def apply_topk(scores, param_map, ratio):
    all_scores = torch.cat(scores)
    k = int(ratio * all_scores.numel())
    _, idx = torch.topk(all_scores, k, largest=False)
    mask = torch.ones_like(all_scores, dtype=torch.bool)
    mask[idx] = False
    pointer = 0
    for (p, numel) in param_map:
        p.data *= mask[pointer:pointer+numel].view_as(p)
        pointer += numel

def magnitude(model, r, **kw):
    scores, pm = [], []
    for n, p in model.named_parameters():
        if 'weight' in n:
            scores.append(p.abs().view(-1))
            pm.append((p, p.numel()))
    apply_topk(scores, pm, r)
    return model

def random_prune(model, r, **kw):
    for n, p in model.named_parameters():
        if 'weight' in n:
            p.data *= (torch.rand_like(p) > r)
    return model

def movement(model, r, train_loader, loss_fn, **kw):
    model.train(); model.zero_grad()
    for i, b in enumerate(train_loader):
        out = model(input_ids=b['input_ids'].to(device),
                    attention_mask=b['attention_mask'].to(device))
        loss_fn(out.logits, b['label'].to(device)).backward()
        if i >= 20: break
    scores, pm = [], []
    for n, p in model.named_parameters():
        if p.grad is not None and 'weight' in n:
            scores.append((-(p * p.grad)).abs().view(-1))
            pm.append((p, p.numel()))
    apply_topk(scores, pm, r)
    return model

def drift_basic(model, r, initial_weights, **kw):
    scores, pm = [], []
    for n, p in model.named_parameters():
        if 'weight' in n:
            scores.append((p - initial_weights[n]).abs().view(-1))
            pm.append((p, p.numel()))
    apply_topk(scores, pm, r)
    return model

def drift_norm(model, r, initial_weights, **kw):
    scores, pm = [], []
    for n, p in model.named_parameters():
        if 'weight' in n:
            d = (p - initial_weights[n]).abs()
            d = d / (d.mean() + 1e-8)
            scores.append(d.view(-1))
            pm.append((p, p.numel()))
    apply_topk(scores, pm, r)
    return model

def drift_no_norm(model, r, initial_weights, **kw):
    scores, pm = [], []
    for n, p in model.named_parameters():
        if 'weight' in n and 'norm' not in n:
            scores.append((p - initial_weights[n]).abs().view(-1))
            pm.append((p, p.numel()))
    apply_topk(scores, pm, r)
    return model

def drift_hybrid(model, r, initial_weights, **kw):
    scores, pm = [], []
    for n, p in model.named_parameters():
        if 'weight' in n and 'norm' not in n:
            d = (p - initial_weights[n]).abs()
            d = d / (d.mean() + 1e-8)
            scores.append((d * p.abs()).view(-1))
            pm.append((p, p.numel()))
    apply_topk(scores, pm, r)
    return model

print('Pruning methods defined.')


In [ ]:
# ── RUN ONE MODEL — collects all metrics ──────────────────

def run_model(model_name, epochs, lr, tag, all_rows):
    print(f'\n{"="*60}')
    print(f'MODEL: {model_name}')
    print(f'{"="*60}')

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if 'gpt2' in model_name:
        tokenizer.pad_token = tokenizer.eos_token

    dataset = load_dataset('ag_news')
    def tok(x):
        return tokenizer(x['text'], padding='max_length',
                         truncation=True, max_length=128)

    train_ds = dataset['train'].map(tok, batched=True)
    test_ds  = dataset['test'].map(tok, batched=True)
    cols = ['input_ids', 'attention_mask', 'label']
    train_ds.set_format(type='torch', columns=cols)
    test_ds.set_format(type='torch',  columns=cols)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=4)
    test_loader  = DataLoader(test_ds,  batch_size=16, num_workers=4)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=4).to(device)
    if 'gpt2' in model_name:
        model.config.pad_token_id = tokenizer.pad_token_id

    initial_weights = {n: p.detach().clone()
                       for n, p in model.named_parameters()}

    optimizer = AdamW(model.parameters(), lr=lr)
    loss_fn   = nn.CrossEntropyLoss()

    # ── Training ────────────────────────────────────────────
    print(f'Fine-tuning {epochs} epoch(s)...')
    train_start = time.time()
    for ep in range(epochs):
        model.train()
        for b in tqdm(train_loader, desc=f'Epoch {ep+1}'):
            out  = model(input_ids=b['input_ids'].to(device),
                         attention_mask=b['attention_mask'].to(device))
            loss = loss_fn(out.logits, b['label'].to(device))
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    training_time = time.time() - train_start
    print(f'Training time: {training_time:.1f}s')

    # ── Baseline ────────────────────────────────────────────
    base_params  = count_params(model)
    base_size_mb = effective_size_mb(model)
    base_met = evaluate(model, test_loader, base_params, base_size_mb)
    base_latency = base_met['latency_s']
    base_met['speedup'] = 1.0

    print(f'BASELINE -> Acc:{base_met["accuracy"]} F1:{base_met["f1"]} '
          f'Size:{base_met["eff_size_mb"]}MB')

    all_rows.append({
        'run_tag': tag, 'model': model_name,
        'method': 'Baseline', 'pruning_pct': 0,
        'Accuracy': base_met['accuracy'],
        'F1 Score': base_met['f1'],
        'Precision': base_met['precision'],
        'Recall': base_met['recall'],
        'Inference Latency (s)': base_met['latency_s'],
        'Throughput (samples/sec)': base_met['throughput_sps'],
        'Training Time (s)': round(training_time, 2),
        'Speedup': 1.0,
        'GPU Memory Usage (MB)': base_met['gpu_memory_mb'],
        'GPU Utilization (%)': base_met['gpu_util_pct'],
        'Energy Consumption (J)': base_met['energy_joules'],
        'Compression Ratio': 1.0,
        'Model Size (MB)': base_met['model_size_mb'],
        'eff_size_mb': base_met['eff_size_mb'],
        'Sparsity Level (%)': 0.0,
        'Bit-width Reduction': 32,
        'Parameter Reduction (%)': 0.0,
        'nonzero_params': base_met['nonzero_params'],
    })

    # ── Pruning experiments ──────────────────────────────────
    methods = {
        'Magnitude':    magnitude,
        'Random':       random_prune,
        'Movement':     movement,
        'Drift-Basic':  drift_basic,
        'Drift-Norm':   drift_norm,
        'Drift-NoNorm': drift_no_norm,
        'Drift-Hybrid': drift_hybrid,
    }
    ratios = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

    for mname, fn in methods.items():
        print(f'\n  -- {mname} --')
        for r in ratios:
            m = copy.deepcopy(model)
            m = fn(m, r,
                   train_loader=train_loader,
                   loss_fn=loss_fn,
                   initial_weights=initial_weights)
            met = evaluate(m, test_loader, base_params, base_size_mb, base_latency)
            print(f'    {int(r*100)}% -> Acc:{met["accuracy"]} F1:{met["f1"]} '
                  f'Sparsity:{met["sparsity_pct"]}% '
                  f'Size:{met["eff_size_mb"]}MB '
                  f'Speedup:{met["speedup"]}x')
            all_rows.append({
                'run_tag': tag, 'model': model_name,
                'method': mname, 'pruning_pct': int(r*100),
                'Accuracy': met['accuracy'],
                'F1 Score': met['f1'],
                'Precision': met['precision'],
                'Recall': met['recall'],
                'Inference Latency (s)': met['latency_s'],
                'Throughput (samples/sec)': met['throughput_sps'],
                'Training Time (s)': round(training_time, 2),
                'Speedup': met['speedup'],
                'GPU Memory Usage (MB)': met['gpu_memory_mb'],
                'GPU Utilization (%)': met['gpu_util_pct'],
                'Energy Consumption (J)': met['energy_joules'],
                'Compression Ratio': met['compression_ratio'],
                'Model Size (MB)': met['model_size_mb'],
                'eff_size_mb': met['eff_size_mb'],
                'Sparsity Level (%)': met['sparsity_pct'],
                'Bit-width Reduction': 32,
                'Parameter Reduction (%)': met['param_reduction_pct'],
                'nonzero_params': met['nonzero_params'],
            })
            del m; torch.cuda.empty_cache()

print('run_model() function defined.')


In [ ]:
# ── RUN BOTH MODELS ───────────────────────────────────────
# Results go into all_rows list, then saved to CSV and Excel

all_rows = []

# DistilBERT — 1 epoch
run_model('distilbert-base-uncased',
          epochs=1, lr=2e-5, tag='run1', all_rows=all_rows)

# GPT-2 — 1 epoch
run_model('gpt2',
          epochs=1, lr=2e-5, tag='run1', all_rows=all_rows)

print(f'\nTotal rows collected: {len(all_rows)}')


In [ ]:
# ── SAVE TO CSV AND EXCEL ─────────────────────────────────

df = pd.DataFrame(all_rows)

# Save CSV
df.to_csv('pruning_results_ALL.csv', index=False)
print('Saved: pruning_results_ALL.csv')

# Save Excel — same format as Department_Server_Metrics_Assignment.xlsx
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule

wb = Workbook()
ws = wb.active
ws.title = 'Department_Server_Metrics'

# Title row
ws.merge_cells('A1:V1')
c = ws['A1']
c.value = 'Department Server Evaluation Metrics'
c.font = Font(name='Arial', bold=True, size=13, color='FFFFFF')
c.fill = PatternFill('solid', fgColor='1F3864')
c.alignment = Alignment(horizontal='center', vertical='center')
ws.row_dimensions[1].height = 24

# Column headers
headers = [
    'run_tag','model','method','pruning_pct',
    'Accuracy','F1 Score','Precision','Recall',
    'Inference Latency (s)','Throughput (samples/sec)','Training Time (s)','Speedup',
    'GPU Memory Usage (MB)','GPU Utilization (%)','Energy Consumption (J)',
    'Compression Ratio','Model Size (MB)','eff_size_mb',
    'Sparsity Level (%)','Bit-width Reduction','Parameter Reduction (%)','nonzero_params'
]
thin = Side(style='thin', color='BBBBBB')
bt   = Border(left=thin,right=thin,top=thin,bottom=thin)

for ci, h in enumerate(headers, 1):
    c = ws.cell(row=2, column=ci, value=h)
    c.font      = Font(name='Arial', bold=True, color='FFFFFF', size=9)
    c.fill      = PatternFill('solid', fgColor='2E75B6')
    c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    c.border    = bt
ws.row_dimensions[2].height = 36

# Data rows
MODEL_CLR = {
    'distilbert-base-uncased': 'DEEAF1',
    'gpt2':                    'E2EFDA',
    'microsoft/deberta-v3-base':'EAD1DC',
}
for ri, row in enumerate(all_rows, 3):
    bg = MODEL_CLR.get(row['model'], 'F2F2F2')
    if row['method'] == 'Baseline': bg = 'FFF2CC'
    for ci, h in enumerate(headers, 1):
        v = row.get(h, '')
        c = ws.cell(row=ri, column=ci, value=v)
        c.font      = Font(name='Arial', size=9, bold=(row['method']=='Baseline'))
        c.fill      = PatternFill('solid', fgColor=bg)
        c.alignment = Alignment(horizontal='center', vertical='center')
        c.border    = bt
    ws.row_dimensions[ri].height = 14

# Column widths
widths = [8,28,14,10,10,9,10,9,18,20,16,10,16,14,18,14,13,12,14,12,18,14]
for ci, w in enumerate(widths, 1):
    ws.column_dimensions[get_column_letter(ci)].width = w
ws.freeze_panes = 'E3'

# Color scale on Accuracy
last = len(all_rows) + 2
ws.conditional_formatting.add(f'E3:E{last}',
    ColorScaleRule(start_type='num',start_value=0.25,start_color='FF0000',
                   mid_type='num',  mid_value=0.75, mid_color='FFFF00',
                   end_type='num',  end_value=1.0,  end_color='00B050'))

wb.save('Department_Server_Metrics_Assignment.xlsx')
print('Saved: Department_Server_Metrics_Assignment.xlsx')
print(f'\nTotal experiments: {len(all_rows)}')
print(df[['model','method','pruning_pct','Accuracy','F1 Score','eff_size_mb']].to_string(index=False))


In [ ]:
# ── QUICK VERIFICATION ────────────────────────────────────
print('=== RESULTS SUMMARY ===')
for model_name in df['model'].unique():
    print(f'\n{model_name}:')
    sub = df[df['model']==model_name]
    baseline = sub[sub['method']=='Baseline'].iloc[0]
    print(f'  Baseline: Acc={baseline["Accuracy"]:.4f} Size={baseline["eff_size_mb"]:.2f}MB')
    best = sub.loc[sub['Accuracy'].idxmax()]
    print(f'  Best:     Acc={best["Accuracy"]:.4f} Method={best["method"]} '
          f'Pruning={best["pruning_pct"]}%')
    print(f'  Methods tested: {sub["method"].nunique()}')
    print(f'  Total rows:     {len(sub)}')
print('\nFiles produced:')
print('  pruning_results_ALL.csv')
print('  Department_Server_Metrics_Assignment.xlsx')
